## RBG video yolo format .mp4 and .mat (dataset that earlier we got)

## final for IR and RGB labeling 

In [ ]:
import os
import cv2
import pandas as pd
import random
from tqdm import tqdm

# ================= CONFIG =================
VIDEO_DIR = r"D:\IITBHU Internship\other datasets\Drone-detection-dataset-v1.0.0\DroneDetectionThesis-Drone-detection-dataset-e7a6eaf\Data\Video_V"  # change if want to use IR to there respecive folder
CSV_DIR   = os.path.join(VIDEO_DIR, "YOLO_GT")

OUT_BASE  = r"D:\IITBHU Internship\other datasets\DroneDataset_RGB" # this also change accordingly also if IR then change to DroneDataset_IR

FPS = 30.0
CLASS_ID = 0
TRAIN_RATIO = 0.8
# =========================================

# -------- OUTPUT PATHS --------
paths = {
    "img_tr": os.path.join(OUT_BASE, "images/train"),
    "img_va": os.path.join(OUT_BASE, "images/val"),
    "lbl_tr": os.path.join(OUT_BASE, "labels/train"),
    "lbl_va": os.path.join(OUT_BASE, "labels/val"),
}

for p in paths.values():
    os.makedirs(p, exist_ok=True)

samples = []

# -------- TIME PARSER (CRITICAL) --------
def parse_time_sec(t):
    """
    Converts:
    '0 sec', '0.03333 sec', '1.2 s' → float seconds
    """
    return float(str(t).replace("sec", "").replace("s", "").strip())

# -------- PROCESS ONE VIDEO --------
def process_video(video_path, csv_path):
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        print(f"[ERROR] Cannot open {video_path}")
        return

    df = pd.read_csv(csv_path)
    base = os.path.splitext(os.path.basename(video_path))[0]

    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    for idx, row in df.iterrows():
        try:
            t = parse_time_sec(row["Time"])
            frame_id = int(round(t * FPS))
            frame_id = max(0, min(frame_id, total_frames - 1))

            x = float(row["DRONE_1"])
            y = float(row["DRONE_2"])
            w = float(row["DRONE_3"])
            h = float(row["DRONE_4"])
        except:
            continue

        cap.set(cv2.CAP_PROP_POS_FRAMES, frame_id)
        ret, frame = cap.read()
        if not ret:
            continue

        H, W = frame.shape[:2]

        # ---------- YOLO FORMAT ----------
        xc = (x + w / 2.0) / W
        yc = (y + h / 2.0) / H
        wn = w / W
        hn = h / H

        # Clamp to [0,1]
        if not (0 < wn <= 1 and 0 < hn <= 1):
            continue

        xc = min(max(xc, 0.0), 1.0)
        yc = min(max(yc, 0.0), 1.0)

        img_name = f"{base}_{frame_id:06d}.jpg"
        label = f"{CLASS_ID} {xc:.6f} {yc:.6f} {wn:.6f} {hn:.6f}"

        samples.append((frame, img_name, label))

    cap.release()

# -------- PROCESS ALL VIDEOS --------
video_files = sorted([
    f for f in os.listdir(VIDEO_DIR)
    if f.endswith(".mp4") and f.startswith("V_DRONE")  # change if want to use IR to IR_DRONE
])

for file in tqdm(video_files, desc="Processing videos"):
    base = file.replace(".mp4", "")
    csv_file = f"{base}_DRONE.csv"

    video_path = os.path.join(VIDEO_DIR, file)
    csv_path   = os.path.join(CSV_DIR, csv_file)

    if not os.path.exists(csv_path):
        print(f"[SKIP] Missing CSV for {file}")
        continue

    process_video(video_path, csv_path)

# -------- TRAIN / VAL SPLIT --------
random.shuffle(samples)
split_idx = int(len(samples) * TRAIN_RATIO)

train_samples = samples[:split_idx]
val_samples   = samples[split_idx:]

def save_subset(data, img_dir, lbl_dir):
    for frame, name, label in data:
        cv2.imwrite(os.path.join(img_dir, name), frame)
        with open(os.path.join(lbl_dir, name.replace(".jpg", ".txt")), "w") as f:
            f.write(label)

save_subset(train_samples, paths["img_tr"], paths["lbl_tr"])
save_subset(val_samples,   paths["img_va"], paths["lbl_va"])

print(f"\n[DONE]")
print(f"Train samples: {len(train_samples)}")
print(f"Val samples  : {len(val_samples)}")


Processing videos:  75%|███████▍  | 85/114 [05:15<01:47,  3.71s/it]


error: OpenCV(4.12.0) D:\a\opencv-python\opencv-python\opencv\modules\core\src\alloc.cpp:73: error: (-4:Insufficient memory) Failed to allocate 983040 bytes in function 'cv::OutOfMemoryError'
